# 로컬 직군별 모델 성능 비교
**baseline_42features · stage_4 최종 평가**  
직군 분류: KECO 대분류 기준 6개 그룹 (cepil 제공, 2026-08-23)

In [ ]:
# ── 공통 임포트 & 경로 설정 ────────────────────────────────────────────────
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import numpy as np
import pandas as pd

# 환경(VS Code vs Jupyter)에 따른 루트 디렉토리 자동 감지
current_dir  = Path().resolve()
ROOT         = current_dir.parents[1] if current_dir.name == 'cepil' else current_dir
MODELING_DIR = ROOT / 'data/result/baseline_42features/modeling'
OUT_DIR      = ROOT / 'sandbox/cepil/figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('모델링 경로:', MODELING_DIR)
print('저장 폴더:', OUT_DIR)

In [ ]:
# ── DESIGN-apple.md 토큰 (색상 팔레트) ────────────────────────────────────
# 직군별 색상: apple primary → 계열 6색
# 낮음 그룹 (1, 3): ink / ink-muted  |  양호 그룹 (4, 5): primary
# 보통 그룹 (2): body-muted  |  주의 그룹 (6): canvas-parchment

C = {
    'primary'   : '#0066cc',  # Action Blue
    'primary_d' : '#0071e3',  # Focus Blue
    'ink'       : '#1d1d1f',  # Near-black
    'ink_m80'   : '#333333',
    'ink_m48'   : '#7a7a7a',
    'canvas'    : '#ffffff',
    'parchment' : '#f5f5f7',
    'hairline'  : '#e0e0e0',
    'tile_dark' : '#272729',
    'on_dark'   : '#ffffff',
    'sky_blue'  : '#2997ff',
}

# 그룹별 색상 (성능 수준 반영)
# G1: 낮음-주황계  G2: 보통-하늘  G3: 낮음-주황  G4: 양호-파랑  G5: 양호-파랑  G6: 주의-회색
BAR_COLORS   = ['#E07B39', C['sky_blue'], '#E07B39', C['primary'], C['primary_d'], C['ink_m48']]
BAR_COLORS_A = [c + 'CC' for c in BAR_COLORS]  # 80% alpha (hex alpha)

GROUPS = {
    'group1': '경영·사무·금융',
    'group2': '연구·공학·산업기술',
    'group3': '교육·법률·사회·공공',
    'group4': '보건·의료',
    'group5': '예술·디자인·방송·스포츠',
    'group6': '서비스·영업·판매·운송',
}
GROUP_KEYS = list(GROUPS.keys())
SHORT       = [f'G{i+1}' for i in range(6)]
LABELS_2L   = [f'G{i+1}\n{v}' for i, v in enumerate(GROUPS.values())]

print('디자인 토큰 로드 완료')

In [ ]:
# ── 한글 폰트 자동 설정 ────────────────────────────────────────────────────
def set_korean_font():
    candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic',
                  'NanumBarunGothic', 'Noto Sans CJK KR']
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams['font.family'] = name
            print(f'폰트: {name}')
            return name
    print('한글 폰트를 찾지 못했습니다. 폰트 설치 필요.')
    return None

set_korean_font()
plt.rcParams['axes.unicode_minus'] = False

# Apple 스타일 전역 설정
plt.rcParams.update({
    'axes.facecolor'    : C['canvas'],
    'figure.facecolor'  : C['parchment'],
    'axes.edgecolor'    : C['hairline'],
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.spines.left'  : False,
    'axes.spines.bottom': True,
    'axes.grid'         : True,
    'axes.grid.axis'    : 'y',
    'grid.color'        : C['hairline'],
    'grid.linewidth'    : 0.7,
    'grid.alpha'        : 0.8,
    'xtick.color'       : C['ink_m48'],
    'ytick.color'       : C['ink_m48'],
    'xtick.bottom'      : False,
    'ytick.left'        : False,
    'text.color'        : C['ink'],
})

In [ ]:
# ── 데이터 로드 ────────────────────────────────────────────────────────────
def load_summary():
    rows = []
    for gkey in GROUP_KEYS:
        df   = pd.read_csv(MODELING_DIR / f'stage_4_local_{gkey}/final_test_summary.csv')
        best = df.loc[df['test_f1'].idxmax()].copy()
        best['group'] = gkey
        rows.append(best)
    return pd.DataFrame(rows).reset_index(drop=True)

def load_ci(summary):
    rows = []
    for gkey in GROUP_KEYS:
        df  = pd.read_csv(MODELING_DIR / f'stage_4_local_{gkey}/final_test_bootstrap_ci.csv')
        cand = summary.loc[summary['group'] == gkey, 'candidate'].values[0]
        row  = df[df['candidate'] == cand].iloc[0].copy()
        row['group'] = gkey
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)

def load_confusion(summary):
    result = {}
    for gkey in GROUP_KEYS:
        with open(MODELING_DIR / f'stage_4_local_{gkey}/final_test_confusion_matrices.json', encoding='utf-8') as f:
            data = json.load(f)
        cand = summary.loc[summary['group'] == gkey, 'candidate'].values[0]
        result[gkey] = np.array(data[cand])
    return result

def load_importance(model='xgboost'):
    prefix = 'xgboost' if model == 'xgboost' else 'logistic_regression'
    return {
        gkey: pd.read_csv(MODELING_DIR / f'stage_4_local_{gkey}/{prefix}_final_permutation_importance.csv')
        for gkey in GROUP_KEYS
    }

summary    = load_summary()
ci         = load_ci(summary)
cm_data    = load_confusion(summary)
importance = load_importance('xgboost')

print('데이터 로드 완료')
summary[['group', 'candidate', 'test_f1', 'test_roc_auc', 'test_precision', 'test_recall']]

---
## Chart 1 — F1 Score + Bootstrap 95% CI

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.5), facecolor=C['parchment'])
ax.set_facecolor(C['canvas'])

x      = np.arange(6)
f1     = summary['test_f1'].values
ci_lo  = f1 - ci['ci95_lower'].values
ci_hi  = ci['ci95_upper'].values - f1

# 배경 줄무늬 (alternating parchment / white)
for i in range(6):
    ax.axvspan(i - 0.45, i + 0.45, alpha=0.03 if i % 2 == 0 else 0,
               color=C['ink'], zorder=0)

# 막대
bars = ax.bar(x, f1, width=0.52, color=BAR_COLORS, alpha=0.88,
              zorder=3, linewidth=0)

# Bootstrap CI 에러바
ax.errorbar(x, f1, yerr=[ci_lo, ci_hi],
            fmt='none', color=C['ink_m80'], capsize=7,
            capthick=1.8, linewidth=1.8, zorder=5)

# F1 수치 라벨 (막대 위)
for i, (v, hi) in enumerate(zip(f1, ci_hi)):
    ax.text(x[i], v + hi + 0.018, f'{v:.3f}',
            ha='center', va='bottom',
            fontsize=11.5, fontweight='600',
            color=C['ink'])

# CI 하한값 (에러바 아래 작게)
for i, (v, lo) in enumerate(zip(f1, ci_lo)):
    ax.text(x[i], v - lo - 0.022, f'▾{ci["ci95_lower"].values[i]:.3f}',
            ha='center', va='top', fontsize=8,
            color=C['ink_m48'])

# 기준선 F1=0.5
ax.axhline(0.5, color=C['ink_m48'], linestyle='--',
           linewidth=1.2, alpha=0.7, zorder=2)
ax.text(5.55, 0.502, 'F1 = 0.5', fontsize=9,
        color=C['ink_m48'], va='bottom')

# 성능 등급 배지
badges = ['⚠ 낮음', '● 보통', '⚠ 낮음', '✓ 양호', '✓ 양호', '⚠ n 과소']
badge_c = ['#E07B39', C['sky_blue'], '#E07B39', C['primary'], C['primary'], C['ink_m48']]
for i, (b, bc) in enumerate(zip(badges, badge_c)):
    ax.text(x[i], 0.06, b, ha='center', fontsize=8.5,
            color=bc, fontweight='600')

ax.set_xticks(x)
ax.set_xticklabels(LABELS_2L, fontsize=10)
ax.set_ylim(0, 0.88)
ax.set_ylabel('F1 Score', fontsize=11, color=C['ink_m80'])
ax.set_title('직군별 F1 Score + Bootstrap 95% CI',
             fontsize=15, fontweight='600',
             color=C['ink'], pad=18, loc='left')
ax.tick_params(axis='x', pad=6)

# 범례 (색상 의미)
legend_items = [
    mpatches.Patch(color='#E07B39', alpha=0.88, label='낮음 (F1 < 0.5)'),
    mpatches.Patch(color=C['sky_blue'], alpha=0.88, label='보통'),
    mpatches.Patch(color=C['primary'], alpha=0.88, label='양호 (F1 ≥ 0.65)'),
    mpatches.Patch(color=C['ink_m48'], alpha=0.88, label='표본 과소'),
]
ax.legend(handles=legend_items, loc='upper right',
          framealpha=0.9, edgecolor=C['hairline'],
          fontsize=9)

fig.tight_layout(pad=2)
fig.savefig(OUT_DIR / 'chart1_f1_ci.png', dpi=180, bbox_inches='tight',
            facecolor=C['parchment'])
plt.show()

---
## Chart 2 — ROC-AUC & Avg Precision 비교

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6), facecolor=C['parchment'])
ax.set_facecolor(C['canvas'])

x   = np.arange(6)
roc = summary['test_roc_auc'].values
ap  = summary['test_average_precision'].values
w   = 0.34

# ROC-AUC 막대 (진한)
b1 = ax.bar(x - w/2 - 0.02, roc, width=w,
            color=BAR_COLORS, alpha=0.90, zorder=3, label='ROC-AUC')
# Avg Precision 막대 (연한)
b2 = ax.bar(x + w/2 + 0.02, ap, width=w,
            color=BAR_COLORS, alpha=0.42, zorder=3, label='Avg Precision',
            edgecolor=BAR_COLORS, linewidth=1.1)

# 수치 라벨
for i in range(6):
    ax.text(x[i] - w/2 - 0.02, roc[i] + 0.012,
            f'{roc[i]:.3f}', ha='center', va='bottom',
            fontsize=10, fontweight='600', color=C['ink'])
    ax.text(x[i] + w/2 + 0.02, ap[i] + 0.012,
            f'{ap[i]:.3f}', ha='center', va='bottom',
            fontsize=9.5, color=C['ink_m80'])

# 기준선
ax.axhline(0.5, color=C['ink_m48'], linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(5.6, 0.502, 'Random\n= 0.5', fontsize=8, color=C['ink_m48'])

ax.set_xticks(x)
ax.set_xticklabels(LABELS_2L, fontsize=10)
ax.set_ylim(0, 0.86)
ax.set_ylabel('Score', fontsize=11, color=C['ink_m80'])
ax.set_title('ROC-AUC vs Average Precision',
             fontsize=15, fontweight='600', color=C['ink'], pad=18, loc='left')

ax.legend(loc='upper right', framealpha=0.9,
          edgecolor=C['hairline'], fontsize=10)

fig.tight_layout(pad=2)
fig.savefig(OUT_DIR / 'chart2_roc_ap.png', dpi=180,
            bbox_inches='tight', facecolor=C['parchment'])
plt.show()

---
## Chart 3 — Precision / Recall 비교 (선형)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6), facecolor=C['parchment'])
ax.set_facecolor(C['canvas'])

x    = np.arange(6)
prec = summary['test_precision'].values
rec  = summary['test_recall'].values
acc  = summary['test_accuracy'].values

# 영역 채우기 (Precision-Recall 간격)
ax.fill_between(x, prec, rec,
                alpha=0.07, color=C['primary'],
                zorder=1)

# 선 + 포인트
ax.plot(x, rec,  'o-', color=C['sky_blue'],  linewidth=2.5,
        markersize=9, markeredgecolor=C['canvas'],
        markeredgewidth=1.5, zorder=4, label='Recall')
ax.plot(x, prec, 's-', color='#E07B39', linewidth=2.5,
        markersize=9, markeredgecolor=C['canvas'],
        markeredgewidth=1.5, zorder=4, label='Precision')
ax.plot(x, acc,  '^--', color=C['ink_m48'], linewidth=1.5,
        markersize=7, alpha=0.7, zorder=3, label='Accuracy')

# 수치 라벨
for i in range(6):
    ax.text(x[i] + 0.06, rec[i]  + 0.015, f'{rec[i]:.2f}',
            fontsize=10, color=C['sky_blue'], fontweight='600')
    ax.text(x[i] + 0.06, prec[i] - 0.03,  f'{prec[i]:.2f}',
            fontsize=10, color='#E07B39', fontweight='600')

# G4 Recall 강조 주석
ax.annotate('G4 Recall 82%\n(보건·의료)',
            xy=(3, rec[3]), xytext=(3.35, rec[3] - 0.13),
            arrowprops=dict(arrowstyle='->', color=C['primary'],
                            lw=1.5, connectionstyle='arc3,rad=-0.2'),
            fontsize=9, color=C['primary'], fontweight='600')

ax.set_xticks(x)
ax.set_xticklabels(LABELS_2L, fontsize=10)
ax.set_ylim(0.2, 1.0)
ax.set_ylabel('Score', fontsize=11, color=C['ink_m80'])
ax.set_title('Precision / Recall / Accuracy',
             fontsize=15, fontweight='600', color=C['ink'], pad=18, loc='left')
ax.legend(loc='upper left', framealpha=0.9,
          edgecolor=C['hairline'], fontsize=10)

fig.tight_layout(pad=2)
fig.savefig(OUT_DIR / 'chart3_prec_rec.png', dpi=180,
            bbox_inches='tight', facecolor=C['parchment'])
plt.show()

---
## Chart 4 — Confusion Matrix (6개 그룹)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10), facecolor=C['parchment'])
fig.suptitle('Confusion Matrix — 최적 모델 (threshold = 0.5)',
             fontsize=16, fontweight='600', color=C['ink'],
             y=0.97, x=0.06, ha='left')

for idx, (gkey, label) in enumerate(GROUPS.items()):
    ax  = axes[idx // 3][idx % 3]
    cm  = cm_data[gkey]  # [[TN, FP], [FN, TP]]
    col = BAR_COLORS[idx]

    # TN/TP 어두운 색, FP/FN 연한 색
    cell_colors = [
        [col, '#F0F0F0'],   # row 0: [TN(정답), FP(오탐)]
        ['#F5F5F5', col],   # row 1: [FN(누락), TP(정답)]
    ]

    for i in range(2):
        for j in range(2):
            bg_c = col if (i == j) else C['parchment']
            alpha_val = 0.85 if (i == j) else 1.0
            bg_c_actual = col if (i == j) else '#EBEBED'

            rect = FancyBboxPatch(
                (j + 0.04, 1 - i + 0.04), 0.92, 0.92,
                boxstyle='round,pad=0.02',
                facecolor=bg_c_actual if i == j else '#EBEBED',
                alpha=0.88 if i == j else 0.9,
                edgecolor=C['hairline'], linewidth=1,
                transform=ax.transData, zorder=2
            )
            ax.add_patch(rect)

            label_map = {(0,0):'TN', (0,1):'FP', (1,0):'FN', (1,1):'TP'}
            txt_col   = C['on_dark'] if i == j else C['ink_m80']
            ax.text(j + 0.5, 1 - i + 0.62,
                    label_map[(i,j)],
                    ha='center', va='center',
                    fontsize=11, color=txt_col,
                    alpha=0.75 if i != j else 1.0)
            ax.text(j + 0.5, 1 - i + 0.38,
                    str(cm[i, j]),
                    ha='center', va='center',
                    fontsize=22, fontweight='600',
                    color=txt_col)

    # Precision / Recall
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    prec_val = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec_val  = tp / (tp + fn) if (tp + fn) > 0 else 0

    ax.set_facecolor(C['canvas'])
    ax.set_xlim(0, 2)
    ax.set_ylim(0, 2)
    ax.set_xticks([0.5, 1.5])
    ax.set_yticks([0.5, 1.5])
    ax.set_xticklabels(['미취업 예측', '취업 예측'], fontsize=9.5)
    ax.set_yticklabels(['취업', '미취업'], fontsize=9.5)
    ax.tick_params(left=False, bottom=False)
    ax.set_xlabel(f'Precision = {prec_val:.2f}   Recall = {rec_val:.2f}',
                  fontsize=10, color=C['ink_m80'], labelpad=8)
    ax.set_title(f'G{idx+1}  {label}',
                 fontsize=12, fontweight='600',
                 color=C['ink'], pad=10, loc='left')
    ax.spines[:].set_visible(False)
    ax.grid(False)

    # 가로/세로 구분선
    ax.axhline(1, color=C['canvas'], linewidth=3, zorder=3)
    ax.axvline(1, color=C['canvas'], linewidth=3, zorder=3)

    # 실제 레이블 (y축)
    ax.text(-0.12, 1.5, '실제\n미취업', ha='right', va='center',
            fontsize=9, color=C['ink_m48'])
    ax.text(-0.12, 0.5, '실제\n취업', ha='right', va='center',
            fontsize=9, color=C['ink_m48'])

fig.tight_layout(pad=2.5)
fig.savefig(OUT_DIR / 'chart4_confusion.png', dpi=180,
            bbox_inches='tight', facecolor=C['parchment'])
plt.show()

---
## Chart 5 — Feature Importance (XGBoost, Top 10)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor=C['parchment'])
fig.suptitle('XGBoost Feature Importance (Permutation, Top 10)',
             fontsize=16, fontweight='600', color=C['ink'],
             y=0.98, x=0.03, ha='left')

# 피처명 한글화 (주요 피처만)
FEAT_KO = {
    'age'                         : 'age (나이)',
    'student_status'              : 'student_status (재학상태)',
    'graduation_prep_experience'  : 'graduation_prep_experience',
    'major_group'                 : 'major_group (전공계열)',
    'months_since_graduation'     : 'months_since_graduation',
    'gender'                      : 'gender (성별)',
    'region_5'                    : 'region_5 (거주지역)',
    'currently_preparing_exam'    : 'currently_preparing_exam',
    'baseline_year'               : 'baseline_year (조사연도)',
    'recent_job_search'           : 'recent_job_search',
    'education_level'             : 'education_level (학력)',
    'has_employment_certificate'  : 'has_employment_certificate',
    'has_major_related_certificate': 'has_major_related_cert',
    'prep_effort_08'              : 'prep_effort_08',
    'prep_effort_04'              : 'prep_effort_04',
    'prep_effort_03'              : 'prep_effort_03',
    'nonemployment_type'          : 'nonemployment_type',
    'recent_employment_prep'      : 'recent_employment_prep',
    'student_type'                : 'student_type',
    'university_type'             : 'university_type',
    'has_certificate'             : 'has_certificate',
}

for idx, (gkey, glabel) in enumerate(GROUPS.items()):
    ax  = axes[idx // 3][idx % 3]
    df  = importance[gkey].copy()
    col = BAR_COLORS[idx]

    # 상위 10개만
    top10 = df.head(10).copy()
    feat_labels = [FEAT_KO.get(f, f) for f in top10['feature']]

    y_pos  = np.arange(len(top10))
    vals   = top10['importance_mean'].values
    errs   = top10['importance_std'].values
    b_cols = [col if v >= 0 else '#CCCCCC' for v in vals]

    ax.set_facecolor(C['canvas'])
    bars_h = ax.barh(y_pos, vals, color=b_cols, alpha=0.88,
                     xerr=errs, error_kw={'ecolor': C['ink_m48'],
                                          'capsize': 3, 'linewidth': 0.8},
                     zorder=3)
    ax.axvline(0, color=C['ink_m48'], linewidth=0.8, zorder=4)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(feat_labels, fontsize=8.2)
    ax.invert_yaxis()
    ax.set_title(f'G{idx+1}  {glabel}',
                 fontsize=11.5, fontweight='600',
                 color=C['ink'], loc='left', pad=8)
    ax.set_xlabel('F1 drop (양수=유용, 음수=노이즈)', fontsize=8.5,
                  color=C['ink_m48'])
    ax.tick_params(left=False)
    ax.spines['bottom'].set_color(C['hairline'])
    ax.grid(axis='x', alpha=0.4, color=C['hairline'])
    ax.grid(axis='y', visible=False)

    # 수치 라벨
    for i, v in enumerate(vals):
        ha  = 'left' if v >= 0 else 'right'
        off = 0.0008 if v >= 0 else -0.0008
        c   = C['ink'] if v >= 0 else C['ink_m48']
        ax.text(v + off, i, f'{v:.3f}',
                va='center', ha=ha, fontsize=7.5, color=c)

fig.tight_layout(pad=2.5, h_pad=3.5)
fig.savefig(OUT_DIR / 'chart5_importance.png', dpi=180,
            bbox_inches='tight', facecolor=C['parchment'])
plt.show()

---
## Chart 6 — 종합 Dashboard (PPT용 1장)

In [ ]:
# PPT 슬라이드 1장 크기(16:9 기준)
fig = plt.figure(figsize=(20, 11.25), facecolor=C['parchment'])
gs  = gridspec.GridSpec(
    2, 3,
    figure=fig,
    left=0.06, right=0.97,
    top=0.88, bottom=0.10,
    hspace=0.55, wspace=0.30
)

# 타이틀
fig.text(0.06, 0.94,
         '로컬 직군별 모델 성능 — 종합 Dashboard',
         fontsize=18, fontweight='600',
         color=C['ink'])
fig.text(0.06, 0.905,
         'baseline_42features · stage_4 최종 평가 · 직군 분류: KECO 대분류 (cepil, 2026-08-23)',
         fontsize=10, color=C['ink_m48'])

x = np.arange(6)
f1  = summary['test_f1'].values
roc = summary['test_roc_auc'].values
ap  = summary['test_average_precision'].values
prec= summary['test_precision'].values
rec = summary['test_recall'].values
ci_lo = f1 - ci['ci95_lower'].values
ci_hi = ci['ci95_upper'].values - f1

# ── (0,0) F1 + CI ─────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(C['canvas'])
ax1.bar(x, f1, width=0.55, color=BAR_COLORS, alpha=0.88, zorder=3)
ax1.errorbar(x, f1, yerr=[ci_lo, ci_hi],
             fmt='none', color=C['ink_m80'],
             capsize=5, linewidth=1.5, zorder=5)
for i, (v, hi) in enumerate(zip(f1, ci_hi)):
    ax1.text(x[i], v + hi + 0.02, f'{v:.3f}',
             ha='center', fontsize=9, fontweight='600', color=C['ink'])
ax1.axhline(0.5, color=C['ink_m48'], linestyle='--', linewidth=1, alpha=0.6)
ax1.set_xticks(x)
ax1.set_xticklabels(SHORT, fontsize=9)
ax1.set_ylim(0, 0.85)
ax1.set_title('F1 + 95% CI', fontsize=12, fontweight='600',
              color=C['ink'], loc='left')
ax1.spines['left'].set_visible(False)
ax1.spines['bottom'].set_color(C['hairline'])

# ── (0,1) ROC-AUC ────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(C['canvas'])
ax2.bar(x - 0.18, roc, width=0.33, color=BAR_COLORS, alpha=0.90, zorder=3, label='ROC-AUC')
ax2.bar(x + 0.18, ap,  width=0.33, color=BAR_COLORS, alpha=0.40, zorder=3, label='Avg Prec.',
        edgecolor=BAR_COLORS, linewidth=0.8)
for i in range(6):
    ax2.text(x[i]-0.18, roc[i]+0.012, f'{roc[i]:.3f}',
             ha='center', fontsize=8.5, fontweight='600', color=C['ink'])
ax2.axhline(0.5, color=C['ink_m48'], linestyle='--', linewidth=1, alpha=0.6)
ax2.set_xticks(x)
ax2.set_xticklabels(SHORT, fontsize=9)
ax2.set_ylim(0, 0.85)
ax2.set_title('ROC-AUC vs Avg Precision', fontsize=12, fontweight='600',
              color=C['ink'], loc='left')
ax2.legend(fontsize=8, framealpha=0.8)
ax2.spines['left'].set_visible(False)
ax2.spines['bottom'].set_color(C['hairline'])

# ── (0,2) Precision / Recall ──────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(C['canvas'])
ax3.fill_between(x, prec, rec, alpha=0.06, color=C['primary'])
ax3.plot(x, rec,  'o-', color=C['sky_blue'], lw=2.2,
         markersize=7, markeredgecolor=C['canvas'], markeredgewidth=1.2,
         label='Recall')
ax3.plot(x, prec, 's-', color='#E07B39', lw=2.2,
         markersize=7, markeredgecolor=C['canvas'], markeredgewidth=1.2,
         label='Precision')
for i in range(6):
    ax3.text(x[i]+0.08, rec[i]+0.015,  f'{rec[i]:.2f}',
             fontsize=8, color=C['sky_blue'], fontweight='600')
    ax3.text(x[i]+0.08, prec[i]-0.03,  f'{prec[i]:.2f}',
             fontsize=8, color='#E07B39')
ax3.set_xticks(x)
ax3.set_xticklabels(SHORT, fontsize=9)
ax3.set_ylim(0.2, 1.0)
ax3.set_title('Precision / Recall', fontsize=12, fontweight='600',
              color=C['ink'], loc='left')
ax3.legend(fontsize=8, framealpha=0.8)
ax3.spines['left'].set_visible(False)
ax3.spines['bottom'].set_color(C['hairline'])

# ── (1, 0-2) Feature Importance 3개 그룹 (G3·G4·G5 — 가장 흥미로운 것) ─
highlight_groups = ['group3', 'group4', 'group5']
hi_labels = {
    'group3': 'G3 교육·법률·사회·공공  (낮음)',
    'group4': 'G4 보건·의료  (양호)',
    'group5': 'G5 예술·디자인·방송·스포츠  (양호)',
}

for col_idx, gkey in enumerate(highlight_groups):
    ax = fig.add_subplot(gs[1, col_idx])
    ax.set_facecolor(C['canvas'])
    df   = importance[gkey].head(10)
    vals = df['importance_mean'].values
    feat = [FEAT_KO.get(f, f) for f in df['feature']]
    yp   = np.arange(len(df))
    bc   = [BAR_COLORS[GROUP_KEYS.index(gkey)] if v >= 0 else '#CCCCCC' for v in vals]

    ax.barh(yp, vals, color=bc, alpha=0.88, zorder=3)
    ax.axvline(0, color=C['ink_m48'], linewidth=0.8)
    ax.set_yticks(yp)
    ax.set_yticklabels(feat, fontsize=7.8)
    ax.invert_yaxis()
    ax.set_title(hi_labels[gkey], fontsize=10.5, fontweight='600',
                 color=C['ink'], loc='left')
    ax.set_xlabel('F1 drop', fontsize=8, color=C['ink_m48'])
    ax.tick_params(left=False)
    ax.spines['bottom'].set_color(C['hairline'])
    ax.spines['left'].set_visible(False)
    ax.grid(axis='x', alpha=0.3)
    ax.grid(axis='y', visible=False)

    for i, v in enumerate(vals):
        ha  = 'left' if v >= 0 else 'right'
        off = 0.001 if v >= 0 else -0.001
        ax.text(v + off, i, f'{v:.3f}',
                va='center', ha=ha, fontsize=7, color=C['ink_m48'])

# 하단 범례 / 주석
fig.text(0.06, 0.03,
         '● 색상: 주황=낮음  하늘=보통  파랑=양호  회색=표본과소  '
         '| Feature importance: 양수=예측에 유용, 음수=노이즈  '
         '| 하단 차트: G3·G4·G5 (대조 비교)',
         fontsize=8.5, color=C['ink_m48'])

fig.savefig(OUT_DIR / 'chart6_dashboard_ppt.png', dpi=180,
            bbox_inches='tight', facecolor=C['parchment'])
plt.show()
print('저장 완료:', OUT_DIR / 'chart6_dashboard_ppt.png')

In [ ]:
# 저장 파일 목록 확인
for f in sorted(OUT_DIR.glob('*.png')):
    print(f'{f.name:45s}  {f.stat().st_size/1024:.0f} KB')